# 06 — Replica exchange and a comparison of methods

**Learning goals**

- run temperature replica exchange (T-REMD) and follow walkers through the temperature ladder,
- see how ladder spacing controls exchange acceptance,
- run Hamiltonian replica exchange (H-REMD) with a barrier-softening ladder,
- place all methods of this tutorial side by side — as different strategies, not as a ranking.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time as timer
import inspect
import warnings
import numpy as np
import matplotlib.pyplot as plt

from src.potentials import double_well_force, double_well_landmarks, true_free_energy
from src.langevin import brownian_dynamics
from src import replica_exchange as rx
from src.metadynamics import run_wt_metadynamics, metad_free_energy
from src.umbrella import run_umbrella_windows
from src.wham import wham_1d
from src import accelerated_md as amd
from src.analysis import histogram_free_energy, free_energy_rmse, count_transitions
from src.plotting import (set_style, save_figure, plot_reference, free_energy_axes, animate_replicas,
                          show_animation, PALETTE, REFERENCE, MUTED, METHOD_COLORS)
set_style()

## Parameters

In [ ]:
a, b = 2.0, 0.5
T = 0.4                                     # physical temperature (reference replica)
friction, dt = 1.0, 0.002
n_steps = 100_000                           # per replica (200 time units)
exchange_stride = 50                        # steps between exchange attempts
stride = 10

temperatures = rx.geometric_temperatures(T, 1.6, 4)   # T-REMD ladder, temperatures[0] = T
poor_temperatures = np.array([T, 0.45, 0.5, 6.0])     # ladder experiment: one large gap
lambda_values = np.array([1.0, 0.6, 0.3, 0.1])        # H-REMD ladder, lambda_values[0] = 1 (physical)

bins = np.linspace(-1.8, 1.7, 71)
seed = 2026
MAKE_ANIMATION = False                      # optional replica animation (Section 7)

## 1. Temperature replica exchange

$R$ copies of the system run at $T_1 = T < T_2 < \dots < T_R$. Every `exchange_stride` steps, neighbouring
temperatures attempt to swap configurations, alternating the pairs $(0,1),(2,3),\dots$ and $(1,2),(3,4),\dots$. A swap of
$x_i$ (at $\beta_i$) and $x_j$ (at $\beta_j$) is accepted with

$$P_\mathrm{acc} = \min\!\left[1, \exp\!\big((\beta_i-\beta_j)\,[V(x_i)-V(x_j)]\big)\right].$$

The joint ensemble of all replicas has weight $\prod_k e^{-\beta_k V(x_k)}$; this acceptance rule satisfies
detailed balance with respect to it, so every replica — in particular the lowest one — keeps sampling its own
canonical distribution. Our dynamics are overdamped, so no velocities need rescaling after a swap; inertial MD codes
must treat momenta (and thermostats) consistently, e.g. by scaling velocities by $\sqrt{T_\mathrm{new}/T_\mathrm{old}}$.

In [ ]:
print(inspect.getsource(rx.tremd_log_acceptance))
print(inspect.getsource(rx.metropolis_accept))

## 2. Run T-REMD

In [ ]:
lm = double_well_landmarks(a, b)
core = (lm["x_left"] + 0.3, lm["x_right"] - 0.3)
xc = 0.5 * (bins[1:] + bins[:-1])
F_ref = true_free_energy(xc, a, b)
region = F_ref <= lm["barrier_from_left"] + 2 * T

def summarize(res, label):
    R = len(res["temperatures"])
    trips = [rx.count_round_trips(res["walker_slot"][:, w], R) for w in range(R)]
    print(f"{label}: neighbour acceptance {np.round(res['acceptance'], 2)}, round trips per walker {trips}")

t0 = timer.perf_counter()
tremd = rx.run_replica_exchange(lm["x_left"], temperatures, 1.0, n_steps, exchange_stride, dt,
                                np.random.default_rng(seed), a=a, b=b, friction=friction, stride=stride)
time_tremd = timer.perf_counter() - t0
print("temperatures:", np.round(temperatures, 3))
summarize(tremd, "T-REMD")

In [ ]:
def plot_ladder_traces(ax, res, labels, walker=0):
    for w in range(res["walker_slot"].shape[1]):
        ax.step(res["exchange_time"], res["walker_slot"][:, w], where="post", lw=0.4, color=MUTED, alpha=0.5)
    ax.step(res["exchange_time"], res["walker_slot"][:, walker], where="post", lw=1.2, color=PALETTE[0],
            label=f"walker {walker}")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("time")
    ax.legend(loc="upper right", fontsize=8)

fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
plot_ladder_traces(axes[0], tremd, [f"T={t:.2f}" for t in temperatures])
axes[0].set_xlim(0, 20)
axes[0].set_xlabel("")
axes[0].set_title("replica index of each walker (first 20 time units)", fontsize=10)
axes[1].plot(tremd["time"], tremd["x_slot"][:, 0], lw=0.6, color=METHOD_COLORS["tremd"])
axes[1].set_ylabel(rf"$x$ in the $T$ = {T} slot")
axes[1].set_xlabel("time")
fig.tight_layout()
save_figure(fig, "06_tremd_walkers")
plt.show()

Walkers diffuse up and down the ladder. A configuration that crossed the barrier at high temperature can be passed
down to the reference temperature, so the reference slot switches basins much more often than unbiased dynamics
would. Note that the slot trace (lower panel) is *not* a continuous trajectory: it jumps whenever a swap is accepted.
Round-trip counts from one short run are only a rough indicator of ladder mixing.

## 3. Is the reference ensemble sampled correctly?

In [ ]:
t0 = timer.perf_counter()
plain = brownian_dynamics(lm["x_left"], lambda x: double_well_force(x, a, b), n_steps, dt, T,
                          np.random.default_rng(seed), friction, stride=stride)
time_plain = timer.perf_counter() - t0
w_exact = np.exp(-F_ref / T)
p_right_exact = w_exact[xc > lm["x_barrier"]].sum() / w_exact.sum()
walkers = rx.walker_trajectories(tremd)

print(f"P(right basin) exact {p_right_exact:.3f} | plain BD {np.mean(plain > lm['x_barrier']):.3f} | "
      f"T-REMD reference slot {np.mean(tremd['x_slot'][:, 0] > lm['x_barrier']):.3f}")
print(f"basin switches seen at T = {T}: plain BD {count_transitions(plain, *core)}, "
      f"T-REMD slot 0 {count_transitions(tremd['x_slot'][:, 0], *core)}")
print("physical transitions along each (continuous) walker trajectory:", count_transitions(walkers, *core))

## 4. Temperature ladder experiment

Same number of replicas, different spacing. A large gap between two temperatures means their energy
distributions barely overlap, so swaps across the gap are rarely accepted and the ladder is cut in two.

In [ ]:
poor = rx.run_replica_exchange(lm["x_left"], poor_temperatures, 1.0, n_steps, exchange_stride, dt,
                               np.random.default_rng(seed), a=a, b=b, friction=friction, stride=stride)
summarize(poor, "poorly spaced ")
summarize(tremd, "geometric     ")

fig, ax = plt.subplots(figsize=(6, 3))
pairs = np.arange(len(temperatures) - 1)
ax.bar(pairs - 0.2, tremd["acceptance"], width=0.4, color=METHOD_COLORS["tremd"], label=f"geometric {np.round(temperatures, 2)}")
ax.bar(pairs + 0.2, poor["acceptance"], width=0.4, color=MUTED, label=f"poorly spaced {poor_temperatures}")
ax.set_xticks(pairs, [f"({i},{i + 1})" for i in pairs])
ax.set_xlabel("neighbouring pair")
ax.set_ylabel("acceptance")
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

With a single coordinate the energy distributions are broad, so even the large gap still accepts some swaps.
In a system with many degrees of freedom the energy distributions narrow relative to their separation and the
same gap would be nearly impassable — Section 6 makes this concrete.

## 5. Hamiltonian replica exchange

All replicas run at the same temperature $T$, but with different Hamiltonians. Here a pedagogical barrier-softening
ladder scales only the double-well term:

$$U_\lambda(x) = \lambda\,a(x^2-1)^2 + bx, \qquad 0 < \lambda \le 1,$$

with the physical reference at $\lambda=1$. Swapping $x_i \leftrightarrow x_j$ between Hamiltonians $U_i, U_j$ is accepted with

$$P_\mathrm{acc} = \min[1, e^{-\Delta}], \qquad \Delta = \beta\left[U_i(x_j)+U_j(x_i)-U_i(x_i)-U_j(x_j)\right].$$

This ladder is a teaching device, not a general prescription for designing biomolecular H-REMD.

In [ ]:
x_plot = np.linspace(-1.7, 1.6, 300)
fig, ax = plt.subplots(figsize=(6, 3.2))
ramp = plt.matplotlib.colors.LinearSegmentedColormap.from_list("lam", ["#86b6ef", "#0d366b"])
for i, lam in enumerate(lambda_values):
    U = rx.ladder_potential(x_plot[:, None], lam, a, b)
    ax.plot(x_plot, U - U.min(), color=ramp(i / (len(lambda_values) - 1)), label=rf"$\lambda$ = {lam}")
ax.set_ylim(-0.2, 4)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$U_\lambda(x) - \min U_\lambda$")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

t0 = timer.perf_counter()
hremd = rx.run_replica_exchange(lm["x_left"], np.full(len(lambda_values), T), lambda_values, n_steps,
                                exchange_stride, dt, np.random.default_rng(seed), a=a, b=b,
                                friction=friction, stride=stride)
time_hremd = timer.perf_counter() - t0
summarize(hremd, "H-REMD")
print(f"P(right basin) in the lambda = 1 slot: {np.mean(hremd['x_slot'][:, 0] > lm['x_barrier']):.3f} "
      f"(exact {p_right_exact:.3f})")

## 6. T-REMD versus H-REMD

| | T-REMD | H-REMD |
|---|---|---|
| replicas differ in | temperature | Hamiltonian |
| what is "heated" | every energetic degree of freedom | only the term that is scaled |
| reference replica | lowest temperature | $\lambda = 1$ |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
plot_ladder_traces(axes[0], tremd, [f"T={t:.2f}" for t in temperatures])
plot_ladder_traces(axes[1], hremd, [f"λ={l}" for l in lambda_values])
axes[0].set_title("T-REMD", fontsize=10)
axes[1].set_title("H-REMD", fontsize=10)
for ax in axes:
    ax.set_xlim(0, 20)
fig.tight_layout()
plt.show()

**Hidden degrees of freedom.** Add `n_bath` decoupled harmonic coordinates (as in Notebook 05) and repeat both
ladders. T-REMD heats the bath too, so the energy distributions of neighbouring temperatures separate and acceptance
falls. This H-REMD ladder scales only the double-well term, so its acceptance does not depend on the bath.

In [ ]:
print(f"{'n_bath':>6}   {'T-REMD acceptance':>22}   {'H-REMD acceptance':>22}")
for n_bath in (0, 8, 32):
    common = dict(a=a, b=b, friction=friction, n_bath=n_bath, stride=stride)
    rt = rx.run_replica_exchange(lm["x_left"], temperatures, 1.0, 30_000, exchange_stride, dt,
                                 np.random.default_rng(seed), **common)
    rh = rx.run_replica_exchange(lm["x_left"], np.full(len(lambda_values), T), lambda_values, 30_000,
                                 exchange_stride, dt, np.random.default_rng(seed), **common)
    print(f"{n_bath:6d}   {str(np.round(rt['acceptance'], 2)):>22}   {str(np.round(rh['acceptance'], 2)):>22}")

In this toy the barrier lives in one known coordinate, so a targeted Hamiltonian ladder is easy to build. In real
systems the choice of what to scale is a design problem, and H-REMD does not automatically need fewer replicas or
converge faster than T-REMD.

## 7. Optional: replica animation

Static traces above are sufficient; set `MAKE_ANIMATION = True` to replay the first 20 time units of T-REMD.
Dot colour marks walker identity, so an accepted swap appears as two colours trading rows.

In [ ]:
if MAKE_ANIMATION:
    pots = [(x_plot, rx.ladder_potential(x_plot[:, None], 1.0, a, b))] * len(temperatures)
    frames = np.arange(0, 2000, 25)
    anim = animate_replicas(tremd, pots, [f"T={t:.2f}" for t in temperatures], frames)
    display(show_animation(anim))

## 8. What each method modifies

| Method | What is modified? | Needs predefined CV? | Multiple replicas? | Direct bias? | Reweighting / reconstruction | Main pedagogical idea |
|---|---|---|---|---|---|---|
| Unbiased BD | nothing | no | no | no | plain histogram | baseline; trapped by barriers |
| WT-MetaD | history-dependent bias along a chosen CV | yes | no (one walker here) | yes, time-dependent | $F = -\frac{\gamma}{\gamma-1}V_\mathrm{bias}$ | fill visited regions, with tempered hills |
| Umbrella + WHAM | multiple restrained ensembles at chosen CV locations | yes | independent windows (no exchange) | yes, static harmonic | WHAM combines windows | overlap connects windows |
| aMD | raises low-potential regions with a smooth boost of $V$ | no | no | yes, on the potential energy | $e^{\beta\Delta V}$ weights | smoothing vs. noisy weights |
| GaMD | constrained harmonic boost of $V$ designed for tractable reweighting | no | no | yes, on the potential energy | cumulant expansion (approx.) | narrow, near-Gaussian boost |
| T-REMD | exchanges configurations between temperatures | no | yes | no | reference replica sampled directly | diffusion in temperature space |
| H-REMD | exchanges configurations between Hamiltonians | no CV, but a choice of what to scale | yes | auxiliary replicas only | reference replica sampled directly | targeted barrier softening |

This table describes the simple versions used here; production variants (multiple-walker MetaD, REST2,
dual-boost GaMD, replica-exchange umbrella sampling, …) blur several of these boundaries.

## 9. Free-energy comparison

Each method is re-run here with the default settings of its own notebook. All profiles use the same bins, are
shifted to $\min F = 0$, and are compared to $F_\mathrm{true}$ only on bins inside the comparison region that the
method actually estimated (coverage is reported). The integration budgets differ between methods.

In [ ]:
force = lambda x: double_well_force(x, a, b)
estimates, diagnostics = {}, {}

def record(name, F, steps, seconds, note=""):
    estimates[name] = F
    rmse, cov = free_energy_rmse(F, F_ref, region)
    diagnostics[name] = (steps, seconds, rmse / T, cov, note)

# WT-MetaD (Notebook 03 defaults)
t0 = timer.perf_counter()
x_grid = np.linspace(-2.5, 2.5, 501)
md_res = run_wt_metadynamics(lm["x_left"], force, 200_000, dt, T, np.random.default_rng(seed), hill_height=0.1,
                             hill_sigma=0.1, deposition_stride=250, bias_factor=8.0, x_grid=x_grid, friction=friction)
F_md = np.interp(xc, x_grid, metad_free_energy(md_res["bias"], 8.0))
record("WT-MetaD", F_md - F_md.min(), 200_000, timer.perf_counter() - t0,
       f"{count_transitions(md_res['x'], *core)} transitions")

# Umbrella sampling + WHAM (Notebook 04 defaults)
t0 = timer.perf_counter()
centers = np.linspace(-1.5, 1.4, 16)
us = run_umbrella_windows(centers, 40.0, force, 2_000, 20_000, dt, T, np.random.default_rng(seed), friction, 10)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    wh = wham_1d(us["samples"], centers, 40.0, T, bins)
record("Umbrella + WHAM", wh["free_energy"], 16 * 22_000, timer.perf_counter() - t0,
       f"WHAM converged={wh['converged']} in {wh['n_iter']} it., warnings={len(caught)}")

# aMD and GaMD (Notebook 05 defaults)
system = dict(n_bath=4, k_bath=10.0, a=a, b=b, friction=friction)
x0 = np.full(16, lm["x_left"])
t0 = timer.perf_counter()
r_amd = amd.run_boosted_dynamics(x0, 60_000, dt, T, np.random.default_rng(seed), method="amd", E=2.0, alpha=0.5, **system)
x_s, dV_s = r_amd["x"][100:].ravel(), r_amd["dV"][100:].ravel()
record("aMD (exp. reweighting)", histogram_free_energy(x_s, bins, T, log_weights=dV_s / T, min_count=20)[1],
       16 * 60_000, timer.perf_counter() - t0, f"{count_transitions(r_amd['x'], *core).sum()} transitions")
t0 = timer.perf_counter()
p_g, _ = amd.calibrate_gamd(x0, 10_000, 3, 10_000, dt, T, np.random.default_rng(seed + 1), sigma0=1.5 * T, **system)
r_g = amd.run_boosted_dynamics(x0, 60_000, dt, T, np.random.default_rng(seed), method="gamd", E=p_g["E"], k=p_g["k"], **system)
x_s, dV_s = r_g["x"][100:].ravel(), r_g["dV"][100:].ravel()
record("GaMD (cumulant)", amd.cumulant_free_energy(x_s, dV_s, bins, T, min_count=20)[1],
       16 * (60_000 + 40_000), timer.perf_counter() - t0, f"{count_transitions(r_g['x'], *core).sum()} transitions")

# Replica exchange: reference replica only
for name, res, sec in [("T-REMD (reference slot)", tremd, time_tremd), ("H-REMD (λ = 1 slot)", hremd, time_hremd)]:
    record(name, histogram_free_energy(res["x_slot"][100:, 0], bins, T, min_count=10)[1],
           len(res["temperatures"]) * n_steps, sec, f"acceptance {np.round(res['acceptance'].min(), 2)}–{np.round(res['acceptance'].max(), 2)}")
record("unbiased BD", histogram_free_energy(plain[100:], bins, T, min_count=10)[1], n_steps, time_plain,
       f"{count_transitions(plain, *core)} transitions")

In [ ]:
colors = {"WT-MetaD": METHOD_COLORS["metad"], "Umbrella + WHAM": METHOD_COLORS["umbrella"],
          "aMD (exp. reweighting)": METHOD_COLORS["amd"], "GaMD (cumulant)": METHOD_COLORS["gamd"],
          "T-REMD (reference slot)": METHOD_COLORS["tremd"], "H-REMD (λ = 1 slot)": METHOD_COLORS["hremd"],
          "unbiased BD": METHOD_COLORS["unbiased"]}
fig, axes = plt.subplots(2, 4, figsize=(13, 5.4), sharex=True, sharey=True)
for ax, (name, F) in zip(axes.ravel(), estimates.items()):
    plot_reference(ax, xc, F_ref, label=None)
    ax.plot(xc, F, color=colors[name], lw=1.6)
    ax.set_title(name, fontsize=10)
    ax.set_ylim(-0.2, 4.2)
axes.ravel()[-1].axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$F$")
for ax in axes[-1, :3]:
    ax.set_xlabel(r"$x$")
axes[0, 3].set_xlabel(r"$x$")
axes[0, 3].xaxis.set_tick_params(labelbottom=True)
fig.suptitle("dashed: $F_\\mathrm{true}$; gaps: no estimate (insufficient samples)", fontsize=10)
fig.tight_layout()
save_figure(fig, "06_free_energy_comparison")
plt.show()

print(f"{'method':24s} {'steps':>9} {'wall s':>7} {'RMSE/kT':>8} {'coverage':>9}   notes")
for name, (steps, sec, err, cov, note) in diagnostics.items():
    print(f"{name:24s} {steps:9,d} {sec:7.1f} {err:8.2f} {cov:9.0%}   {note}")

**How to read this.**

- Methods that bias or flatten the landscape (MetaD, umbrella sampling, aMD, GaMD) give estimates across the
  barrier; under these parameters they approach the known 1D reference within a fraction of $k_BT$.
- The replica-exchange reference replicas sample both basins, but at the physical temperature the barrier top
  itself is visited too rarely to estimate $F$ there — the reference replica is not biased towards it.
- Unbiased BD over the same time mainly samples the starting basin.
- RMSE values depend on the region, the binning, the seed and the budget. They are diagnostics for this toy, not
  a ranking of methods, and they say nothing about which method suits a given biomolecular problem.

## 10. A note on computational cost

Umbrella windows and REMD replicas are naturally parallel; this pure-Python implementation vectorizes or
serializes them on one core. Wall-clock times above are therefore illustrative only and must not be read as
algorithmic benchmarks. The step counts are listed so that the very different budgets are visible.

## Takeaways

- T-REMD lets configurations diffuse through a temperature ladder; the reference replica samples the physical ensemble.
- Exchange acceptance is controlled by the overlap of neighbouring ensembles; one bad gap can split a ladder.
- H-REMD keeps the temperature and changes the Hamiltonian; only the $\lambda = 1$ replica represents the physical system.
- T-REMD heats every degree of freedom, a targeted Hamiltonian ladder only the chosen term — useful, but a design choice.
- The methods attack the sampling problem in fundamentally different ways:

```text
Integrator                  controls how equations of motion are discretized
Thermostat / barostat       controls thermodynamic conditions
Ordinary MD / BD            may become trapped in metastable states
WT-Metadynamics             adds history-dependent bias along a CV
Umbrella Sampling           samples predefined restrained windows
WHAM                        combines biased windows into an unbiased distribution
aMD                         smooths low-energy basins with a boost potential
GaMD                        uses a constrained harmonic boost designed for tractable reweighting
T-REMD                      lets configurations diffuse through a temperature ladder
H-REMD                      lets configurations diffuse through a Hamiltonian ladder
```

*References:* Sugita & Okamoto, *Chem. Phys. Lett.* **314**, 141 (1999), doi:10.1016/S0009-2614(99)01123-9 ·
Fukunishi, Watanabe & Takada, *J. Chem. Phys.* **116**, 9058 (2002), doi:10.1063/1.1472510 (Hamiltonian REMD).